##connect drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Extract dataset

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/MusicGenreSorter/genres_original.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("استخراج تموم شد!")
print(os.listdir(extract_path))

استخراج تموم شد!
['genres_original']


##install package

In [ ]:
!pip install librosa -q

## Step 1: Load Audio Files and Extract Features

In [ ]:
import librosa
import numpy as np
import os

dataset_path = '/content/dataset/genres_original'
genres = os.listdir(dataset_path)
print("ژانرها:", genres)

# خوندن یک فایل نمونه
sample_genre = genres[0]
sample_folder = os.path.join(dataset_path, sample_genre)
sample_file = os.listdir(sample_folder)[0]
sample_path = os.path.join(sample_folder, sample_file)

y, sr = librosa.load(sample_path, sr=22050)
print(f"نمونه فایل: {sample_file}")
print(f"طول سیگنال: {len(y)}, نرخ نمونه‌برداری: {sr}")

ژانرها: ['reggae', 'country', 'jazz', 'blues', 'disco', 'classical', 'pop', 'metal', 'rock', 'hiphop']
نمونه فایل: reggae.00060.wav
طول سیگنال: 661504, نرخ نمونه‌برداری: 22050


## Step 2: Extract Features from Full Dataset

In [ ]:
features = []
labels = []

for genre in genres:
    genre_path = os.path.join(dataset_path, genre)
    for file in os.listdir(genre_path):
        file_path = os.path.join(genre_path, file)
        try:
            y, sr = librosa.load(file_path, sr=22050, duration=30)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            mel_db = librosa.power_to_db(mel_spec, ref=np.max)
            features.append(mel_db)
            labels.append(genre)
        except:
            print(f"خطا در فایل: {file_path}")

print(f"تعداد کل نمونه‌ها: {len(features)}")

/tmp/ipykernel_666/91407019.py:9: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=22050, duration=30)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


خطا در فایل: /content/dataset/genres_original/jazz/jazz.00054.wav
تعداد کل نمونه‌ها: 999


## Step 3: Prepare Data for Training

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = max(f.shape[1] for f in features)

features_padded = np.array([
    np.pad(f, ((0,0),(0, max_len - f.shape[1])), mode='constant')
    for f in features
])

le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

print("شکل نهایی داده‌ها:", features_padded.shape)
print("تعداد کلاس‌ها:", len(le.classes_))

شکل نهایی داده‌ها: (999, 128, 1292)
تعداد کلاس‌ها: 10


## Step 3.5: Normalize Features

In [ ]:
mean = features_padded.mean()
std = features_padded.std()

features_normalized = (features_padded - mean) / std

print("میانگین قبل:", features_padded.mean(), "بعد:", features_normalized.mean())
print("انحراف معیار بعد:", features_normalized.std())

میانگین قبل: -42.790268 بعد: 1.1955166e-05
انحراف معیار بعد: 1.0000005


## Step 4: Split Data into Train and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_normalized, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (799, 128, 1292, 1)
Test: (200, 128, 1292, 1)


## Step 5: Build the CNN Model

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(128, 1292, 1)),

    layers.Conv2D(16, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 1290, 16)  │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 1290, 16)  │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 645, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 643, 32)    │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 643, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 321, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 319, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 319, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 159, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,554 (111.54 KB)

 Trainable params: 28,330 (110.66 KB)

 Non-trainable params: 224 (896.00 B)

## Step 6: Train the Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 14s 150ms/step - accuracy: 0.2353 - loss: 2.1391 - val_accuracy: 0.1050 - val_loss: 2.2687
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.3054 - loss: 1.8591 - val_accuracy: 0.1000 - val_loss: 2.6302
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.3579 - loss: 1.7296 - val_accuracy: 0.1000 - val_loss: 3.1837
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.4068 - loss: 1.6428 - val_accuracy: 0.1000 - val_loss: 3.7663
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.4068 - loss: 1.6681 - val_accuracy: 0.1350 - val_loss: 3.5655
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.4418 - loss: 1.5397 - val_accuracy: 0.1050 - val_loss: 3.7576
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.4606 - loss: 1.5319 - val_accuracy: 0.1800 - val_loss: 2.9360
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.4831 - loss: 1.4130 - val_accuracy: 0.2450 -

## Step 7: Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"دقت نهایی روی داده تست: {test_acc:.2%}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7500 - loss: 0.8270
دقت نهایی روی داده تست: 75.00%


## Step 8: Save the Trained Model

In [ ]:
model.save('/content/drive/MyDrive/MusicGenreSorter/genre_model.keras')
print("مدل ذخیره شد!")

مدل ذخیره شد!


## Step 9: Save the Label Encoder and Normalization Values

In [ ]:
import pickle

with open('/content/drive/MyDrive/MusicGenreSorter/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

norm_params = {'mean': mean, 'std': std, 'max_len': max_len}
with open('/content/drive/MyDrive/MusicGenreSorter/norm_params.pkl', 'wb') as f:
    pickle.dump(norm_params, f)

print("همه‌چیز ذخیره شد!")

همه‌چیز ذخیره شد!


## Step 10: Predict Genre for a New Audio File

In [ ]:
def predict_genre(file_path):
    y, sr = librosa.load(file_path, sr=22050, duration=30)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)

    if mel_db.shape[1] < max_len:
        mel_db = np.pad(mel_db, ((0,0),(0, max_len - mel_db.shape[1])), mode='constant')
    else:
        mel_db = mel_db[:, :max_len]

    mel_db = (mel_db - mean) / std
    mel_db = mel_db[np.newaxis, ..., np.newaxis]

    prediction = model.predict(mel_db, verbose=0)
    genre_index = np.argmax(prediction)
    genre_name = le.inverse_transform([genre_index])[0]
    confidence = np.max(prediction)

    return genre_name, confidence

test_file = '/content/dataset/genres_original/rock/rock.00005.wav'
genre, conf = predict_genre(test_file)
print(f"ژانر پیش‌بینی‌شده: {genre} (اطمینان: {conf:.2%})")

ژانر پیش‌بینی‌شده: rock (اطمینان: 61.64%)


## Step 11: Upload and Test a New Song (Outside Dataset)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
test_file = '/content/audio.mp3'
genre, conf = predict_genre(test_file)
print(f"ژانر پیش‌بینی‌شده: {genre} (اطمینان: {conf:.2%})")

## Classify a Song and Sort It into a Genre Folder

In [ ]:
import os
import shutil

def classify_and_sort(file_path, output_dir='sorted_songs'):
    genre, confidence = predict_genre(file_path)

    genre_folder = os.path.join(output_dir, genre)
    os.makedirs(genre_folder, exist_ok=True)

    filename = os.path.basename(file_path)
    destination = os.path.join(genre_folder, filename)
    shutil.copy(file_path, destination)

    print(f"فایل '{filename}' به‌عنوان '{genre}' تشخیص داده شد (اطمینان: {confidence:.2%})")
    print(f"منتقل شد به: {destination}")

    return genre, confidence